# Fine-Tuning Nemotron Nano 30B as an LLM Router with Tinker SDK

**A Joint Case Study: NVIDIA × Glean**

> *"Tinker is awesome and very easy to use — most things just worked out of the box, reward shaping is the main iteration we needed to do. Basically no hyperparameter tuning needed at all."*
> — Glean Engineering Team

---

## Overview

This tutorial demonstrates how to fine-tune **Nemotron Nano 30B** as a production LLM router using NVIDIA's **Tinker SDK**. Inspired by Glean's real-world agentic deployment, we show how a compact model can be trained to orchestrate tool calls in an agent loop — routing queries to the right tools and deciding when to hand off to a frontier model.

### What You'll Learn

- The **LLM Router pattern**: separating tool-selection planning from text generation
- **Two-phase post-training**: DPO for rapid baseline conditioning + RLVR for reward-optimized routing
- Using **Tinker SDK** with minimal configuration (defaults work well — no hyperparameter sweeps needed)
- Optimal **vLLM serving** configuration for Nemotron Nano on H100s, drawn from Glean's production config
- Adapting the open-source **[Salesforce xLAM Function Calling 60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k)** dataset as a public analog to production routing traces

### Authors

| Organization | Contributors |
|---|---|
| NVIDIA | Vineeth, Farshad, Justin |
| Glean | Eddie, Rahul, Abhi, Thai, Zubin |

### Requirements

- 1× NVIDIA H100 80GB (or A100 80GB) for training and serving
- Python 3.10+
- Access to [Tinker SDK](https://github.com/NVIDIA-NeMo/tinker) and [tinker-cookbook](https://github.com/NVIDIA-NeMo/tinker-cookbook)
- HuggingFace account (for model weights)

## Architecture: The LLM Router Pattern

```
User Query
    │
    ▼
┌──────────────────────────────────────────────┐
│         Nemotron Nano 30B Router             │
│                                              │
│  Input: query + conv history + tool list     │
│                    │                         │
│          ┌─────────▼──────────┐              │
│          │   Agentic Loop     │              │
│          │                    │              │
│          │  [Tool Call(s)] ──►│ Execute      │
│          │        │           │ Tools        │
│          │  [Tool Call(s)] ──►│              │
│          │        │           │              │
│          │  <|tool_stop|>     │              │
│          │     or             │              │
│          │  <|escalate|>      │              │
│          └────────────────────┘              │
└──────────────────┬───────────────────────────┘
                   │  (passes collected tool call
                   │   outputs as context)
                   ▼
       ┌───────────────────────┐
       │   Frontier Model      │
       │   (GPT-4o, Claude,    │
       │    Llama, etc.)       │
       │   Generates final     │
       │   text response       │
       └───────────────────────┘
```

### Key Design Decisions

**1. No text generation in the router.**  
The router only outputs tool calls and termination signals — never prose. All text generation is delegated to the frontier model, which receives the tool call results as context.

**2. Parallel tool calls.**  
The router can issue multiple tool calls simultaneously in a single step (e.g., search across multiple indexes in parallel).

**3. Full context, no prompt distillation.**  
Input always includes the complete query + conversation history + full tool list. Glean explicitly avoids distilling prompts because tool descriptions vary at serving time — distillation would break generalization.

**4. Adaptive escalation.**  
The `<|escalate|>` termination signal tells the orchestrating system that the frontier model needs higher reasoning effort (e.g., extended thinking). The `<|tool_stop|>` signal means the collected context is sufficient for a normal frontier model pass.

**5. Single H100 replica.**  
Nemotron Nano 30B fits on one H100 80GB with no tensor parallelism. This maximizes the number of replicas per compute budget and simplifies deployment.

In [ ]:
# Install required packages
# Tinker SDK is available at https://github.com/NVIDIA-NeMo/tinker
# tinker-cookbook is at https://github.com/NVIDIA-NeMo/tinker-cookbook
!pip install -q \
    "datasets>=2.14.0" \
    "transformers>=4.47.0" \
    "torch>=2.1.0" \
    "vllm>=0.6.0" \
    "peft>=0.9.0" \
    "trl>=0.12.0" \
    "requests>=2.31.0"

# Install Tinker SDK from source
# !pip install -q git+https://github.com/NVIDIA-NeMo/tinker.git
# !pip install -q git+https://github.com/NVIDIA-NeMo/tinker-cookbook.git

In [ ]:
import json
import random
import re
from collections import defaultdict
from typing import Optional

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# Tinker SDK imports
# These follow the patterns established in tinker-cookbook
import tinker
from tinker import TinkerTrainer, LoRAConfig
from tinker.data import Datum
from tinker.losses import DPOLoss
from tinker.envs import SearchEnv
from tinker.inference import TinkerInference

# ── Constants ──────────────────────────────────────────────────────────────────
MODEL_ID   = "nvidia/Nemotron-Nano-30B-Instruct"
DATASET_ID = "Salesforce/xlam-function-calling-60k"

# Termination tokens — emitted by the router to signal end-of-loop state
SUCCESS_TOKEN  = "<|tool_stop|>"   # enough context collected; frontier model can respond
ESCALATE_TOKEN = "<|escalate|>"    # frontier model needs elevated reasoning effort

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"Model  : {MODEL_ID}")
print(f"Dataset: {DATASET_ID}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available'}")

## Dataset: Salesforce xLAM Function Calling 60k

Glean's training data comes from **internal production traces** of Glean Assistant. For this tutorial we use the publicly available [Salesforce xLAM Function Calling 60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) dataset as a drop-in analog. xLAM was developed by Salesforce Research for training next-generation function-calling agents and is a strong fit because:

| Property | xLAM | Glean's approach |
|---|---|---|
| Tool schemas | Full JSON schemas with descriptions | Same |
| Parallel calls | Included | Primary use case |
| Multi-turn | Supported | Required |
| Domain coverage | Web APIs, search, databases, math | Search-heavy |
| Size | 60,000 examples | Internal traces |
| Synthetic data | No (real API traces) | No (no synthetic data) |

### Mapping xLAM to the Router Training Format

| xLAM field | Router interpretation |
|---|---|
| GPT response = tool call JSON | Router should emit that tool call |
| GPT response = plain text | Router should emit `<\|escalate\|>` (frontier model needed) |
| Multiple tool calls in one response | Parallel routing step |
| Conversation history | Full agentic loop context |

In [ ]:
print(f"Loading {DATASET_ID} ...")
raw_dataset = load_dataset(DATASET_ID, split="train")
print(f"Total examples : {len(raw_dataset):,}")
print(f"Columns        : {raw_dataset.column_names}")

# ── Inspect a sample ─────────────────────────────────────────────────────────
sample = raw_dataset[0]
print("\n─── Sample ───────────────────────────────────────────────────────────")
for turn in sample['conversations']:
    role = turn['from'].upper()
    preview = turn['value'][:160].replace('\n', ' ')
    print(f"[{role}] {preview}...")

tools_sample = json.loads(sample['tools'])
print(f"\nTools available ({len(tools_sample)}):")
for t in tools_sample:
    print(f"  {t['name']}: {t['description'][:80]}...")

# ── Dataset statistics ────────────────────────────────────────────────────────
tool_call_count    = 0
text_response_count = 0
parallel_call_count = 0

for ex in raw_dataset:
    gpt_turns = [t for t in ex['conversations'] if t['from'] == 'gpt']
    if not gpt_turns:
        continue
    last_val = gpt_turns[-1]['value'].strip()
    try:
        parsed = json.loads(last_val)
        calls  = parsed if isinstance(parsed, list) else [parsed]
        tool_call_count += 1
        if len(calls) > 1:
            parallel_call_count += 1
    except json.JSONDecodeError:
        text_response_count += 1

total = len(raw_dataset)
print(f"\n─── Statistics ───────────────────────────────────────────────────────")
print(f"Tool call responses : {tool_call_count:,} ({tool_call_count/total*100:.1f}%)")
print(f"  ↳ Parallel calls  : {parallel_call_count:,} ({parallel_call_count/total*100:.1f}%)")
print(f"Text responses      : {text_response_count:,} ({text_response_count/total*100:.1f}%)  ← maps to ESCALATE")

In [ ]:
def build_system_prompt(tools: list[dict]) -> str:
    """Build the router system prompt, injecting the available tool list."""
    return (
        "You are an LLM router operating inside an agentic loop. "
        "Your only responsibility is to decide which tools to call given the "
        "user query and conversation context. You do NOT generate prose responses.\n\n"
        "When you have issued all necessary tool calls, terminate with EXACTLY ONE of:\n"
        f"  {SUCCESS_TOKEN}  — collected context is sufficient for the frontier model\n"
        f"  {ESCALATE_TOKEN} — the query requires elevated reasoning effort\n\n"
        "Tool calls must be valid JSON arrays. Parallel calls are allowed.\n\n"
        f"Available tools:\n{json.dumps(tools, indent=2)}"
    )


def xlam_to_router(example: dict) -> dict | None:
    """Convert one xLAM example into router training format."""
    try:
        tools = json.loads(example['tools'])
    except (json.JSONDecodeError, KeyError):
        return None

    messages = [{"role": "system", "content": build_system_prompt(tools)}]
    has_parallel = False

    for turn in example['conversations']:
        if turn['from'] == 'system':
            continue  # replaced above

        role    = "user" if turn['from'] == 'human' else "assistant"
        content = turn['value']

        if turn['from'] == 'gpt':
            try:
                parsed = json.loads(content)
                calls  = parsed if isinstance(parsed, list) else [parsed]
                if len(calls) > 1:
                    has_parallel = True
                # Normalise to JSON array string
                content = json.dumps(calls)
            except json.JSONDecodeError:
                # Plain text response → escalation in router terms
                content = ESCALATE_TOKEN

        messages.append({"role": role, "content": content})

    # Ensure the last assistant turn ends with a termination token
    asst_turns = [m for m in messages if m['role'] == 'assistant']
    if not asst_turns:
        return None
    last = asst_turns[-1]
    if last['content'] != ESCALATE_TOKEN and SUCCESS_TOKEN not in last['content']:
        last['content'] = last['content'] + "\n" + SUCCESS_TOKEN

    return {
        "id":           example['id'],
        "messages":     messages,
        "tools":        tools,
        "has_parallel": has_parallel,
    }


print("Converting xLAM → router format ...")
router_examples = [r for ex in raw_dataset if (r := xlam_to_router(ex)) is not None]

random.shuffle(router_examples)
n_val          = max(500, int(0.05 * len(router_examples)))
val_examples   = router_examples[:n_val]
train_examples = router_examples[n_val:]

print(f"Converted examples  : {len(router_examples):,}")
print(f"  Train             : {len(train_examples):,}")
print(f"  Validation        : {len(val_examples):,}")
print(f"  Has parallel calls: {sum(1 for e in router_examples if e['has_parallel']):,}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
MAX_LEN   = 65_536   # Tinker trains Nemotron Nano at 64k context


def apply_chat_template(messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


# ── Build a pool of tool names for negative sampling ─────────────────────────
all_tool_names = list({t['name'] for ex in train_examples for t in ex['tools']})


def make_rejected_response(chosen: str, tools: list[dict]) -> str:
    """
    Synthesise a rejected response via one of three strategies:
      0 – substitute a wrong tool name
      1 – flip the termination token (SUCCESS ↔ ESCALATE)
      2 – emit an empty tool-call list
    """
    strategy = random.randint(0, 2)

    if strategy == 0:
        try:
            call_str = chosen.replace(SUCCESS_TOKEN, "").replace(ESCALATE_TOKEN, "").strip()
            calls    = json.loads(call_str)
            calls    = calls if isinstance(calls, list) else [calls]
            wrong    = [{**c, "name": random.choice(
                [n for n in all_tool_names if n != c.get("name", "")]
                or all_tool_names
            )} for c in calls]
            return json.dumps(wrong) + "\n" + SUCCESS_TOKEN
        except (json.JSONDecodeError, TypeError):
            return ESCALATE_TOKEN

    if strategy == 1:
        if ESCALATE_TOKEN in chosen:
            t = random.choice(tools) if tools else {"name": "unknown"}
            return json.dumps([{"name": t['name'], "arguments": {}}]) + "\n" + SUCCESS_TOKEN
        return ESCALATE_TOKEN

    # strategy == 2
    return json.dumps([]) + "\n" + SUCCESS_TOKEN


def example_to_datum(ex: dict) -> Datum | None:
    """
    Convert one router training example to a tinker.Datum DPO pair.

    Context  = all messages except the final assistant turn
    Chosen   = ground-truth router output
    Rejected = synthesised negative output
    """
    msgs = ex['messages']
    if len(msgs) < 3:
        return None

    context_text  = apply_chat_template(msgs[:-1])
    chosen_text   = context_text + msgs[-1]['content']
    rejected_text = context_text + make_rejected_response(msgs[-1]['content'], ex['tools'])

    enc = tokenizer(chosen_text,   return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc_r = tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    ctx_len = len(tokenizer(context_text).input_ids)

    return Datum(
        chosen_ids    = enc.input_ids[0],
        rejected_ids  = enc_r.input_ids[0],
        context_length= ctx_len,
        metadata      = {"has_parallel": ex['has_parallel']},
    )


print("Building tinker.Datum objects from training examples ...")
train_data = [d for ex in train_examples if (d := example_to_datum(ex)) is not None]
val_data   = [d for ex in val_examples[:500] if (d := example_to_datum(ex)) is not None]

print(f"Train Datum objects : {len(train_data):,}")
print(f"Val   Datum objects : {len(val_data):,}")
d0 = train_data[0]
print(f"Sample — chosen len : {d0.chosen_ids.shape[0]} tokens")
print(f"Sample — rejected len: {d0.rejected_ids.shape[0]} tokens")

## Phase 1: DPO Fine-Tuning

Direct Preference Optimization (DPO) is used in the **first phase** to rapidly lift the base model from poor tool-calling behaviour to a usable baseline.

### Why DPO first?

From Glean's experience:
> *"The base model behaves very poorly, and we can quickly condition some improvements with simple DPO/SFT."*

DPO requires no rollouts or environment simulation — just offline preference pairs. This makes it dramatically cheaper and faster than RL for the first phase.

### LoRA Configuration

Following Glean's production settings, we use **Tinker's default LoRA configuration**.
No hyperparameter sweeps were needed — the defaults work:

| Parameter | Value | Note |
|---|---|---|
| Rank (`r`) | 32 | Glean production default |
| Alpha | 64 | 2× rank (Tinker default) |
| Target modules | `all-linear` | All attention + MLP projections |
| Dropout | 0.05 | Standard |

### Loss function

Standard DPO sigmoid loss (`β = 0.1`). Tinker's `DPOConfig` exposes this directly.

In [ ]:
# ── LoRA config — matches Glean's production settings ────────────────────────
lora_config = LoRAConfig(
    r              = 32,            # rank — Glean production default
    lora_alpha     = 64,            # 2× rank
    target_modules = "all-linear",  # all attention + MLP projections
    lora_dropout   = 0.05,
    task_type      = "CAUSAL_LM",
)

# ── DPO config ────────────────────────────────────────────────────────────────
dpo_config = tinker.DPOConfig(
    beta           = 0.1,       # KL penalty — Tinker default
    loss_type      = "sigmoid",  # standard DPO loss
    label_smoothing= 0.0,
    reference_free = False,     # keep reference model for KL
)

# ── Trainer config ────────────────────────────────────────────────────────────
dpo_trainer_config = tinker.TrainerConfig(
    model_id                  = MODEL_ID,
    output_dir                = "./checkpoints/phase1_dpo",
    num_train_epochs          = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate             = 5e-5,
    warmup_ratio              = 0.05,
    lr_scheduler_type         = "cosine",
    logging_steps             = 10,
    save_steps                = 500,
    bf16                      = True,
    max_length                = MAX_LEN,
    lora                      = lora_config,
    dpo                       = dpo_config,
)

print("Initialising Phase 1: DPO trainer ...")
dpo_trainer = TinkerTrainer(
    config     = dpo_trainer_config,
    train_data = train_data,
    val_data   = val_data,
)

print("Starting DPO training ...")
dpo_results = dpo_trainer.train()

print("\n─── Phase 1 DPO Results ─────────────────────────────────────────────")
print(f"Final train loss    : {dpo_results.final_loss:.4f}")
print(f"Reward accuracy     : {dpo_results.reward_accuracy:.4f}")
print(f"Checkpoint          : {dpo_trainer_config.output_dir}")

## Phase 2: RLVR with Reward Shaping

After DPO conditions basic behaviour, **Reinforcement Learning with Verifiable Rewards (RLVR)** optimises for what actually matters in production: did the router retrieve the right documents, and did it make the right escalation call?

### Why RLVR after DPO?

> *"We use true RL to do rollouts and achieve better quality."* — Glean Engineering

DPO is supervised over a fixed dataset. RLVR lets the model explore beyond the training distribution and receive reward signals from an environment, enabling it to discover routing strategies that weren't in the original data.

### Combined Reward Function

```
R_total = α · R_search  +  β · R_termination
```

| Component | What it measures | xLAM analog | Glean's production analog |
|---|---|---|---|
| `R_search` | Did the router call the right tools with the right arguments? | F1 against ground-truth tool calls | F1/recall/precision on documents deemed relevant in historical traces |
| `R_termination` | Did the router make the right escalation decision? | Matches ground-truth termination token | Whether the trace used non-core/head tools (requiring escalation) |

Glean found that **reward shaping is the main iteration surface** — not hyperparameters, not LoRA rank. Tune `α` and `β` based on your production quality goals.

### Tinker `SearchEnv`

Tinker's `SearchEnv` (from tinker-cookbook) provides the scaffolding for RLVR rollouts. We subclass it to inject our router-specific reward logic.

In [ ]:
# ── Reward utilities ──────────────────────────────────────────────────────────

def parse_router_output(text: str) -> tuple[list[dict], str]:
    """Split router output into (tool_calls, termination_token)."""
    termination = ESCALATE_TOKEN if ESCALATE_TOKEN in text else SUCCESS_TOKEN
    call_str    = text.replace(SUCCESS_TOKEN, "").replace(ESCALATE_TOKEN, "").strip()
    try:
        parsed = json.loads(call_str)
        calls  = parsed if isinstance(parsed, list) else [parsed] if isinstance(parsed, dict) else []
    except json.JSONDecodeError:
        calls = []
    return calls, termination


def search_reward(pred_calls: list[dict], gt_calls: list[dict]) -> float:
    """
    F1-based reward on tool selection.

    Mirrors Glean's search-based reward:
      precision/recall/F1 on documents deemed relevant in historical traces.
    Here we proxy document relevance with tool selection correctness.
    """
    if not gt_calls:
        return 1.0 if not pred_calls else -0.5  # correctly predicted "no tools needed"
    if not pred_calls:
        return -1.0

    pred_names = {c.get("name", "") for c in pred_calls}
    gt_names   = {c.get("name", "") for c in gt_calls}

    precision = len(pred_names & gt_names) / len(pred_names)
    recall    = len(pred_names & gt_names) / len(gt_names)
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    # Bonus for argument overlap on matched tools (up to +0.2 per tool)
    arg_bonus = 0.0
    for name in pred_names & gt_names:
        p = next((c for c in pred_calls if c.get("name") == name), None)
        g = next((c for c in gt_calls   if c.get("name") == name), None)
        if p and g:
            p_vals = set(str(v) for v in (p.get("arguments") or {}).values())
            g_vals = set(str(v) for v in (g.get("arguments") or {}).values())
            if g_vals:
                arg_bonus += (len(p_vals & g_vals) / len(g_vals)) * 0.2

    return min(1.0, f1 + arg_bonus)


def termination_reward(pred_term: str, gt_term: str) -> float:
    """Binary reward for correct escalation/success decision."""
    return 1.0 if pred_term == gt_term else -1.0


def router_reward(
    completion:   str,
    ground_truth: dict,
    alpha: float = 0.7,
    beta:  float = 0.3,
) -> float:
    """
    Combined router reward: R = α·R_search + β·R_termination

    alpha controls the weight on tool-selection quality.
    beta  controls the weight on the escalation/success decision.
    Glean's defaults: alpha=0.7, beta=0.3.
    """
    pred_calls, pred_term = parse_router_output(completion)
    gt_calls = ground_truth.get("calls", [])
    gt_term  = ground_truth.get("termination", SUCCESS_TOKEN)

    r_s = search_reward(pred_calls, gt_calls)
    r_t = termination_reward(pred_term, gt_term)
    return alpha * r_s + beta * r_t


# ── Smoke test ────────────────────────────────────────────────────────────────
test_cases = [
    ("Perfect match",
     '[{"name": "search", "arguments": {"query": "foo"}}]\n' + SUCCESS_TOKEN,
     {"calls": [{"name": "search", "arguments": {"query": "foo"}}], "termination": SUCCESS_TOKEN}),
    ("Wrong tool",
     '[{"name": "calculator", "arguments": {}}]\n' + SUCCESS_TOKEN,
     {"calls": [{"name": "search", "arguments": {}}], "termination": SUCCESS_TOKEN}),
    ("Correct escalation",
     ESCALATE_TOKEN,
     {"calls": [], "termination": ESCALATE_TOKEN}),
    ("Wrong termination",
     '[{"name": "search", "arguments": {}}]\n' + SUCCESS_TOKEN,
     {"calls": [{"name": "search", "arguments": {}}], "termination": ESCALATE_TOKEN}),
    ("Partial parallel match",
     '[{"name": "search"}, {"name": "calculator"}]\n' + SUCCESS_TOKEN,
     {"calls": [{"name": "search"}, {"name": "weather"}], "termination": SUCCESS_TOKEN}),
]

print("─── Reward Function Tests ────────────────────────────────────────────")
for name, output, gt in test_cases:
    r = router_reward(output, gt)
    print(f"  {name:<28} → {r:+.3f}")

In [ ]:
class RouterSearchEnv(SearchEnv):
    """
    Router environment — subclasses Tinker's SearchEnv (tinker-cookbook).

    Extends the base environment with a combined search + termination reward,
    mirroring Glean's production reward design.
    """

    def __init__(self, alpha: float = 0.7, beta: float = 0.3, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha
        self.beta  = beta

    def compute_reward(self, rollout: tinker.Rollout) -> float:
        return router_reward(
            completion   = rollout.completion,
            ground_truth = rollout.metadata["ground_truth"],
            alpha        = self.alpha,
            beta         = self.beta,
        )

    def is_terminal(self, rollout: tinker.Rollout) -> bool:
        return SUCCESS_TOKEN in rollout.completion or ESCALATE_TOKEN in rollout.completion


def build_rlvr_prompts(examples: list[dict]) -> list[dict]:
    """Convert router examples into RLVR prompt + ground_truth dicts."""
    out = []
    for ex in examples:
        msgs = ex['messages']
        if len(msgs) < 3:
            continue
        gt_calls, gt_term = parse_router_output(msgs[-1]['content'])
        out.append({
            "prompt":       apply_chat_template(msgs[:-1]),
            "ground_truth": {"calls": gt_calls, "termination": gt_term},
        })
    return out


print("Preparing RLVR prompts ...")
rlvr_prompts = build_rlvr_prompts(train_examples)
print(f"RLVR prompts: {len(rlvr_prompts):,}")

# ── Environment ───────────────────────────────────────────────────────────────
router_env = RouterSearchEnv(
    alpha = 0.7,   # search quality weight
    beta  = 0.3,   # termination quality weight
)

# ── RLVR trainer config — continues from Phase 1 DPO checkpoint ──────────────
rlvr_trainer_config = tinker.TrainerConfig(
    model_id                    = MODEL_ID,
    checkpoint_path             = "./checkpoints/phase1_dpo",
    output_dir                  = "./checkpoints/phase2_rlvr",
    num_train_epochs            = 2,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 1e-5,   # lower LR for RL phase
    warmup_ratio                = 0.02,
    lr_scheduler_type           = "cosine",
    logging_steps               = 10,
    save_steps                  = 250,
    bf16                        = True,
    max_length                  = MAX_LEN,
    lora                        = lora_config,
    # RLVR-specific
    num_rollouts_per_step       = 4,     # rollouts per gradient step
    kl_coeff                    = 0.01,  # KL penalty vs. reference model
    clip_range                  = 0.2,   # PPO/GRPO clip range
)

print("Initialising Phase 2: RLVR trainer ...")
rlvr_trainer = TinkerTrainer(
    config        = rlvr_trainer_config,
    env           = router_env,
    train_prompts = rlvr_prompts,
)

print("Starting RLVR training ...")
rlvr_results = rlvr_trainer.train_rl()

print("\n─── Phase 2 RLVR Results ────────────────────────────────────────────")
print(f"Mean reward         : {rlvr_results.mean_reward:.4f}")
print(f"Search component    : {rlvr_results.component_rewards.get('search', float('nan')):.4f}")
print(f"Termination component: {rlvr_results.component_rewards.get('termination', float('nan')):.4f}")
print(f"Checkpoint          : {rlvr_trainer_config.output_dir}")

## Serving with vLLM on H100

After training, the fine-tuned Nemotron Nano 30B router is deployed with **vLLM** on a single **H100 80GB** (Google Cloud `a3-highgpu-1g`).

### Glean's production numbers

| Metric | Value |
|---|---|
| P50 latency | 250 ms |
| P95 latency | 2.5 s |
| P99 latency | 3.5 s |
| GPU | NVIDIA_H100_80GB / a3-highgpu-1g |
| Tensor parallelism | 1 (single GPU — maximises replica count) |

### Nemotron-specific flags

- **`--trust-remote-code`** is required — Nemotron models include custom ops (Mamba architecture components)
- **`--max-model-len=65536`** matches Tinker's 64k training context
- **`--enable-chunked-prefill`** helps with long routing contexts (full conversation history + all tool descriptions)

> **Tuning note from Glean**: `--max-num-batched-tokens=8192` was carried over from a prior Qwen3 deployment and may not be optimal for Nemotron Nano's tokenizer. Set it to your P95 prompt length for best throughput.

In [ ]:
import subprocess


def build_vllm_command(
    model_path:              str,
    host:                    str   = "0.0.0.0",
    port:                    int   = 8080,
    tensor_parallel_size:    int   = 1,      # single H100; increase for multi-GPU
    gpu_memory_utilization:  float = 0.85,
    max_num_seqs:            int   = 8,
    max_num_batched_tokens:  int   = 8192,   # TODO: tune to your P95 prompt length
    max_model_len:           int   = 65_536, # matches Tinker's 64k training context
) -> list[str]:
    """
    Build the vLLM serve command for Nemotron Nano 30B.

    Based on Glean's production deployment config (H100 80GB / a3-highgpu-1g).
    Achieves P50=250ms, P95=2.5s, P99=3.5s latency.
    """
    return [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        f"--model={model_path}",
        f"--host={host}",
        f"--port={port}",
        f"--tensor-parallel-size={tensor_parallel_size}",
        "--swap-space=0",
        f"--gpu-memory-utilization={gpu_memory_utilization}",
        f"--max-num-seqs={max_num_seqs}",
        "--enable-chunked-prefill",
        # Set to ~P95 prompt length for your traffic distribution.
        # Glean note: this value was ported from Qwen3 and may need tuning for Nemotron.
        f"--max-num-batched-tokens={max_num_batched_tokens}",
        # Tinker trains Nemotron Nano at 64k context window.
        f"--max-model-len={max_model_len}",
        # Required: Nemotron uses custom CUDA ops (Mamba components).
        "--trust-remote-code",
    ]


def call_router(
    query:               str,
    conversation_history: list[dict],
    available_tools:     list[dict],
    server_url:          str = "http://localhost:8080",
) -> dict:
    """
    Call the deployed Nemotron Nano router and parse its output.

    Returns a dict with:
      tool_calls     – list of tool call dicts
      termination    – SUCCESS_TOKEN or ESCALATE_TOKEN
      should_escalate– bool convenience flag
    """
    import requests

    messages = [
        {"role": "system",    "content": build_system_prompt(available_tools)},
        *conversation_history,
        {"role": "user",      "content": query},
    ]

    resp = requests.post(
        f"{server_url}/v1/chat/completions",
        json={
            "model":       "nemotron-nano-router",
            "messages":    messages,
            "max_tokens":  512,
            "temperature": 0.0,   # deterministic routing
            "stop":        [SUCCESS_TOKEN, ESCALATE_TOKEN],
        },
        timeout=30,
    )
    resp.raise_for_status()

    raw     = resp.json()["choices"][0]["message"]["content"]
    calls, term = parse_router_output(raw)

    return {
        "tool_calls":      calls,
        "termination":     term,
        "should_escalate": term == ESCALATE_TOKEN,
        "raw":             raw,
    }


# ── Print the production serving command ──────────────────────────────────────
FINAL_CKPT = "./checkpoints/phase2_rlvr/final"
cmd = build_vllm_command(FINAL_CKPT)

print("Production vLLM command (H100 80GB / a3-highgpu-1g):")
print()
print(cmd[0] + " " + cmd[1] + " " + cmd[2] + " \\")
for arg in cmd[3:]:
    print(f"  {arg} \\")
print()
print("Expected latency: P50=250ms  P95=2.5s  P99=3.5s")

In [ ]:
def offline_evaluate(
    examples:        list[dict],
    checkpoint_path: str,
    n_examples:      int = 500,
    alpha:           float = 0.7,
    beta:            float = 0.3,
) -> dict:
    """
    Offline evaluation using TinkerInference (no vLLM server needed).

    Reports:
      mean_reward          – combined router reward (primary signal)
      tool_f1              – mean F1 on tool selection
      termination_accuracy – correct SUCCESS/ESCALATE rate
      escalation_recall    – recall on examples requiring escalation
    """
    inference = TinkerInference(
        model_id        = MODEL_ID,
        checkpoint_path = checkpoint_path,
        lora_config     = lora_config,
    )

    sample = random.sample(examples, min(n_examples, len(examples)))

    rewards, f1s, term_correct, escl_correct, escl_total = [], [], [], [], []

    for ex in sample:
        msgs = ex['messages']
        if len(msgs) < 3:
            continue

        prompt   = apply_chat_template(msgs[:-1])
        gt_text  = msgs[-1]['content']
        gt_calls, gt_term = parse_router_output(gt_text)

        completion = inference.generate(
            prompt, max_tokens=512, temperature=0.0,
            stop=[SUCCESS_TOKEN, ESCALATE_TOKEN],
        )

        pred_calls, pred_term = parse_router_output(completion)

        r = router_reward(completion, {"calls": gt_calls, "termination": gt_term},
                          alpha=alpha, beta=beta)
        rewards.append(r)
        f1s.append(search_reward(pred_calls, gt_calls))
        term_correct.append(1.0 if pred_term == gt_term else 0.0)

        if gt_term == ESCALATE_TOKEN:
            escl_total.append(1)
            escl_correct.append(1 if pred_term == ESCALATE_TOKEN else 0)

    def _mean(lst): return sum(lst) / len(lst) if lst else float('nan')

    return {
        "n_evaluated":          len(rewards),
        "mean_reward":          _mean(rewards),
        "tool_f1":              _mean(f1s),
        "termination_accuracy": _mean(term_correct),
        "escalation_recall":    _mean(escl_correct) if escl_total else float('nan'),
        "escalation_examples":  len(escl_total),
    }


print("Running evaluation on validation set ...")
results = offline_evaluate(
    examples        = val_examples,
    checkpoint_path = "./checkpoints/phase2_rlvr/final",
)

print()
print("─── Evaluation Results ──────────────────────────────────────────────")
for k, v in results.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")

## Results and Production Learnings

### Glean's Production Numbers

| Metric | Value |
|---|---|
| Training | Phase 1 DPO → Phase 2 RLVR |
| LoRA rank | 32 (Tinker default) |
| GPU | H100 80GB / a3-highgpu-1g |
| Tensor parallelism | 1 |
| P50 latency | 250 ms |
| P95 latency | 2.5 s |
| P99 latency | 3.5 s |
| Training data | Internal production traces only (no synthetic data) |
| Hyperparameter tuning | None needed beyond reward weights |

### Key Takeaways

**1. Start with DPO — it's fast and moves the needle immediately.**  
The base Nemotron Nano model exhibits poor tool-calling behaviour. DPO over a small set of preference pairs quickly conditions it to a usable baseline without any rollout infrastructure.

**2. RLVR is where quality is won — reward shaping is the main lever.**  
Glean's primary iteration was reward design, not model architecture or training hyperparameters. Time spent tuning α/β and refining the reward signal pays back more than hyperparameter sweeps.

**3. Tinker defaults are production-ready.**  
Rank-32 LoRA with default alpha. Standard DPO β=0.1. No sweeps needed. Tinker's defaults are well-calibrated for Nemotron Nano.

**4. Preserve full context; avoid prompt distillation.**  
Including the full query + history + tool list at training time keeps the model generalisable. Distilling prompts hurts when tool descriptions vary at inference time.

**5. One H100 is enough — optimise for replica count, not raw GPU utilisation.**  
Nemotron Nano 30B fits on a single H100 80GB. Keeping tensor parallelism at 1 maximises the number of independently-deployable replicas per compute budget.

**6. vLLM `--max-num-batched-tokens` should be tuned to your P95 prompt length.**  
Glean carried this value over from a prior Qwen3 deployment. Profile your production prompt length distribution and set it accordingly for best throughput on Nemotron.

### Next Steps

- **Reward model**: Train a learned reward model on domain preference data for a richer RLVR signal beyond binary F1.
- **Differential escalation**: Route `<|escalate|>` to different frontier model configurations (e.g., thinking-mode vs. standard) based on the predicted difficulty signal.
- **Batching**: Experiment with higher `--max-num-seqs` under high-concurrency workloads.
- **Nemotron 3 Nano**: As improved base models become available, the same two-phase recipe applies — Phase 1 DPO cost is low enough to re-run for each new checkpoint.

---

## Acknowledgments

This tutorial is based on a joint case study between **NVIDIA** and **[Glean](https://glean.com)**.

**NVIDIA team**: Vineeth, Farshad, Justin  
**Glean team**: Eddie, Rahul, Abhi, Thai, Zubin

For questions about Tinker SDK, see:
- [tinker-cookbook](https://github.com/NVIDIA-NeMo/tinker-cookbook) — reference implementations for DPO, RLVR, and reward environments
- [Nemotron model family](https://huggingface.co/nvidia/Nemotron-Nano-30B-Instruct) on Hugging Face